# Smart Factory Simulator with AI-Based Production Control


## 0. Setup and  data

In [1]:
import sys, os, json, pickle, time
sys.path.insert(0, os.path.join(os.getcwd(), "src"))
import numpy as np, pandas as pd
from scipy.stats import wilcoxon, spearmanr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, r2_score
from sklearn.inspection import permutation_importance
from sfs import *

QUICK = False   # True = fast smoke test (about 7 min)
N_MAIN, N_SWEEP = (4, 3) if QUICK else (20, 8)
LOADS = [0.7, 0.8, 0.9, 1.0]; MAIN_LOAD = 0.9
R = os.path.join(os.getcwd(), "results"); os.makedirs(R, exist_ok=True)
t0 = time.time(); log = lambda *a: print(f"[{time.time()-t0:6.0f}s]", *a, flush=True)

## 1. Data and the machine-failure model

In [2]:
# 1) failure model (machine-condition AI)
df, src = load_ai4i(); log("data:", src, len(df), "rows, failure rate", round(df["Machine failure"].mean(), 4))
fm, (Xt, yt) = train_failure_model(df); pt = fm.predict_proba(Xt)[:, 1]
fpr, tpr, _ = roc_curve(yt, pt)
aux = dict(roc=(fpr, tpr), data_source=src)
meta = dict(data_source=src, n_rows=len(df), fail_auc=roc_auc_score(yt, pt), fail_pr_auc=average_precision_score(yt, pt))

[     0s] data: UCI AI4I 2020 (real) 10000 rows, failure rate 0.0339


## 2. Tardiness-risk model

In [3]:
# 2) tardiness-risk model trained on simulated logs from mixed policies
rng = np.random.default_rng(0)
def collect(seeds, loads):
    rows = []
    for s in seeds:
        for ld in loads:
            pol = dict(rule=rng.choice(CLASSIC), pdm=rng.choice([None, "periodic", "predictive"]), thr=0.2, kk=2.0)
            rows += simulate(CFG, s, ld, pol, fail_model=fm, collect=True)["train"]
    d = pd.DataFrame(rows, columns=["job"] + JF + ["tard"]).dropna(); return d
tr = collect(range(100, 100 + (4 if QUICK else 16)), LOADS[:3]); ho = collect(range(300, 300 + (2 if QUICK else 4)), LOADS[:3])
log("train rows", len(tr), "holdout rows", len(ho))
tm = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.06, max_depth=6, random_state=0).fit(tr[JF], np.log1p(tr["tard"]))
ph = tm.predict(ho[JF]); meta.update(tard_r2_log=r2_score(np.log1p(ho["tard"]), ph), tard_spearman=spearmanr(ph, ho["tard"])[0],
    tard_auc_late=roc_auc_score(ho["tard"] > 0, ph))
sub = ho.sample(min(3000, len(ho)), random_state=0)
pi = permutation_importance(tm, sub[JF], np.log1p(sub["tard"]), n_repeats=3, random_state=0)
aux.update(cal=(ph, np.log1p(ho["tard"].values)), imp=dict(zip(JF, pi.importances_mean)))
log("models:", {k: (round(v, 3) if isinstance(v, float) else v) for k, v in meta.items()})

[   207s] train rows 72392 holdout rows 16225
[   209s] models: {'data_source': 'UCI AI4I 2020 (real)', 'n_rows': 10000, 'fail_auc': 0.988, 'fail_pr_auc': 0.906, 'tard_r2_log': 0.589, 'tard_spearman': np.float64(0.695), 'tard_auc_late': 0.919}


## 3. Tuning every method on validation seeds (fair to baselines)

In [4]:
# 3) tune hyper-parameters of every method on validation seeds (fair to baselines)
import warnings; warnings.filterwarnings("ignore")
VAL = [900, 901, 902]
def score(pol, ld=0.9): return np.mean([simulate(CFG, s, ld, pol, fm, tm)["res"]["mean_tard"] for s in VAL])
kk = min([1.0, 2.0, 4.0], key=lambda k: score(dict(rule="ATC", kk=k)))
pw = min([30, 50, 80], key=lambda w: score(dict(rule="EDD", pdm="periodic", per_wear=w)))
thr = min([0.05, 0.1, 0.2, 0.4], key=lambda x: score(dict(rule="ATC", kk=kk, pdm="predictive", thr=x)))
aa = min([0.0, 0.5, 1.0, 2.0], key=lambda a: score(dict(rule="ML", a=a, kk=kk, pdm="predictive", thr=thr)))
meta.update(atc_kk=kk, periodic_wear=pw, pdm_thr=thr, ml_a=aa); log("tuned: kk, per_wear, thr, a =", kk, pw, thr, aa)

POL = {
 "FIFO": dict(rule="FIFO"), "SPT": dict(rule="SPT"), "EDD": dict(rule="EDD"), "CR": dict(rule="CR"),
 "ATC": dict(rule="ATC", kk=kk),
 "EDD + periodic PM": dict(rule="EDD", pdm="periodic", per_wear=pw),
 "EDD + AI-PdM": dict(rule="EDD", pdm="predictive", thr=thr),
 "ATC + AI-PdM": dict(rule="ATC", kk=kk, pdm="predictive", thr=thr),
 "AI-dispatch": dict(rule="ML", a=aa, kk=kk),
 "AI-dispatch + AI-PdM": dict(rule="ML", a=aa, kk=kk, pdm="predictive", thr=thr)}
def run(loads, seeds):
    out = []
    for ld in loads:
        for s in seeds:
            for name, pol in POL.items():
                out.append(dict(policy=name, load=ld, seed=s, **simulate(CFG, s, ld, pol, fm, tm)["res"]))
        log("load", ld, "done")
    return pd.DataFrame(out)
main = run([MAIN_LOAD], range(1000, 1000 + N_MAIN)); main.to_csv(f"{R}/runs_main.csv", index=False)
sweep = run(LOADS, range(2000, 2000 + N_SWEEP)); sweep.to_csv(f"{R}/runs_sweep.csv", index=False)

[   614s] tuned: kk, per_wear, thr, a = 2.0 50 0.05 2.0
[  1799s] load 0.9 done
[  2153s] load 0.7 done
[  2565s] load 0.8 done
[  3013s] load 0.9 done
[  3522s] load 1.0 done


## 4. Main test, load sweep and statistics

In [5]:
# 4) statistics: mean +- 95% CI, paired Wilcoxon + % improvement vs EDD and vs best classical (ATC)
M = ["mean_tard", "pct_tardy", "p95_tard", "mean_flow", "throughput", "failures", "downtime"]
def ci(x): return 1.96 * x.std(ddof=1) / np.sqrt(len(x))
summ = main.groupby("policy", sort=False)[M].agg(["mean", ci]); summ.columns = [f"{a}_{'ci' if b=='ci' else 'mean'}" for a, b in summ.columns]
summ.round(2).to_csv(f"{R}/summary_main.csv")
st = []
for base in ["EDD", "ATC"]:
    b = main[main.policy == base].set_index("seed")
    for p in POL:
        if p == base: continue
        x = main[main.policy == p].set_index("seed")
        for m in ["mean_tard", "pct_tardy", "mean_flow"]:
            d = (x[m] - b[m]); imp = 100 * (1 - x[m].mean() / b[m].mean())
            try: pv = wilcoxon(d).pvalue
            except Exception: pv = np.nan
            bs = [100 * (1 - x[m].values[i].mean() / b[m].values[i].mean()) for i in (np.random.default_rng(k).integers(0, len(d), len(d)) for k in range(500))]
            st.append(dict(baseline=base, policy=p, metric=m, improvement_pct=imp, ci_lo=np.percentile(bs, 2.5), ci_hi=np.percentile(bs, 97.5), wilcoxon_p=pv))
pd.DataFrame(st).round(4).to_csv(f"{R}/stats_vs_baselines.csv", index=False)
sw = sweep.groupby(["policy", "load"], sort=False)[["mean_tard", "pct_tardy", "mean_flow", "failures"]].mean().round(2); sw.to_csv(f"{R}/summary_sweep.csv")

## 4b. Traces for figures

In [6]:
# 5) traces for figures (one seed, load 0.9): Gantt window + WIP
for nm in ["EDD", "AI-dispatch + AI-PdM"]:
    aux[nm] = simulate(CFG, 1000, MAIN_LOAD, POL[nm], fm, tm, trace=True)
pickle.dump(aux, open(f"{R}/aux.pkl", "wb")); json.dump({k: (float(v) if isinstance(v, (np.floating, float)) else v) for k, v in meta.items()}, open(f"{R}/model_meta.json", "w"), indent=1)
log("DONE"); print(summ[["mean_tard_mean", "pct_tardy_mean", "mean_flow_mean", "failures_mean", "downtime_mean"]].round(1))

[  3556s] DONE
                      mean_tard_mean  pct_tardy_mean  mean_flow_mean  \
policy                                                                 
FIFO                            89.5            46.2           401.0   
SPT                             49.7            16.5           285.4   
EDD                             48.8            35.3           367.1   
CR                              40.9            35.0           368.2   
ATC                             29.9            28.4           343.8   
EDD + periodic PM               70.0            45.0           403.0   
EDD + AI-PdM                     9.7            12.0           269.5   
ATC + AI-PdM                     5.3             7.9           258.2   
AI-dispatch                     31.7            21.2           343.8   
AI-dispatch + AI-PdM             6.2             6.9           260.4   

                      failures_mean  downtime_mean  
policy                                              
FIFO          

## 5. Figures (PNG, 300 dpi, one fixed colour per method)

In [7]:
import os, pickle, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
H = os.getcwd(); R, F = f"{H}/results", f"{H}/figures"; os.makedirs(F, exist_ok=True)
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 150, "savefig.bbox": "tight", "font.family": "DejaVu Sans"})
# one fixed colour per policy family, colour-blind safe (Okabe-Ito), identical in every figure
C = {"FIFO": "#999999", "SPT": "#56B4E9", "EDD": "#0072B2", "CR": "#8FA7BF", "ATC": "#332288",
     "EDD + periodic PM": "#E69F00", "EDD + AI-PdM": "#CC79A7", "ATC + AI-PdM": "#D55E00",
     "AI-dispatch": "#009E73", "AI-dispatch + AI-PdM": "#005C3E"}
main = pd.read_csv(f"{R}/runs_main.csv"); sweep = pd.read_csv(f"{R}/runs_sweep.csv"); aux = pickle.load(open(f"{R}/aux.pkl", "rb"))
order = list(C)
def ci(x): return 1.96 * x.std(ddof=1) / np.sqrt(len(x))
def save(n): plt.savefig(f"{F}/{n}.png", dpi=300); plt.show()

In [8]:
# Fig1 main metrics
fig, ax = plt.subplots(1, 4, figsize=(12, 3.6))
for a, (m, t) in zip(ax, [("mean_tard", "Mean tardiness (min)"), ("pct_tardy", "Tardy jobs (%)"), ("mean_flow", "Mean flow time (min)"), ("failures", "Machine failures (count)")]):
    g = main.groupby("policy")[m]; mu, e = g.mean().reindex(order), g.apply(ci).reindex(order)
    a.barh(range(len(order)), mu, xerr=e, color=[C[p] for p in order], capsize=2, height=.72); a.set_title(t, fontsize=9)
    a.set_yticks(range(len(order))); a.set_yticklabels(order if a is ax[0] else [], fontsize=8); a.invert_yaxis()
plt.tight_layout(); save("fig1_main_results")

In [9]:
# Fig2 load sweep
fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
for a, (m, t) in zip(ax, [("mean_tard", "Mean tardiness (min)"), ("pct_tardy", "Tardy jobs (%)")]):
    for p in order:
        d = sweep[sweep.policy == p].groupby("load")[m].agg(["mean", ci])
        a.errorbar(d.index, d["mean"], d["ci"], color=C[p], marker="o", ms=3, lw=1.6 if "AI" in p else 1, label=p, capsize=2)
    a.set_xlabel("Relative load"); a.set_ylabel(t)
ax[0].set_yscale("symlog", linthresh=10); ax[1].legend(fontsize=6.5, frameon=False, bbox_to_anchor=(1.02, 1)); plt.tight_layout(); save("fig2_load_sweep")

In [10]:
# Fig3 ablation: % improvement of each AI component vs EDD and ATC (paired, per seed)
fig, ax = plt.subplots(figsize=(6.5, 3.4)); piv = main.pivot(index="seed", columns="policy", values="mean_tard")
names = ["EDD + periodic PM", "EDD + AI-PdM", "ATC + AI-PdM", "AI-dispatch", "AI-dispatch + AI-PdM"]; w = .38
for k, base in enumerate(["EDD", "ATC"]):
    imp = [100 * (1 - piv[p] / piv[base]) for p in names]
    ax.bar(np.arange(len(names)) + (k - .5) * w, [i.mean() for i in imp], w, yerr=[ci(i) for i in imp], capsize=2, color=[C[p] for p in names], alpha=1 if k == 0 else .55, edgecolor="k", lw=.5, label=f"vs {base}" )
ax.axhline(0, color="k", lw=.6); ax.set_xticks(range(len(names))); ax.set_xticklabels([n.replace(" + ", "\n+ ") for n in names], fontsize=7)
ax.set_ylabel("Tardiness reduction (%)"); ax.legend(["solid: vs EDD", "light: vs ATC"], frameon=False, fontsize=7); save("fig3_ablation")

In [11]:
# Fig4 predictive models: failure ROC + tardiness predictor calibration
fig, ax = plt.subplots(1, 2, figsize=(7.5, 3.3)); fpr, tpr = aux["roc"]
ax[0].plot(fpr, tpr, color=C["AI-dispatch"], lw=2); ax[0].plot([0, 1], [0, 1], "k:", lw=.8); ax[0].set(xlabel="False-positive rate", ylabel="True-positive rate", title="Machine-failure model (held-out)")
p, y = aux["cal"]; q = pd.qcut(p, 12, duplicates="drop"); d = pd.DataFrame({"p": p, "y": y}).groupby(q).mean()
ax[1].plot(d.p, d.y, "o-", color=C["ATC + AI-PdM"]); lo, hi = d.values.min(), d.values.max(); ax[1].plot([lo, hi], [lo, hi], "k:", lw=.8)
ax[1].set(xlabel="Predicted log(1+tardiness)", ylabel="Observed log(1+tardiness)", title="Tardiness-risk model (held-out)"); plt.tight_layout(); save("fig4_model_quality")

In [12]:
# Fig5 permutation importance
imp = pd.Series(aux["imp"]).sort_values(); fig, ax = plt.subplots(figsize=(4.5, 3)); ax.barh(imp.index, imp.values, color=C["AI-dispatch"])
ax.set_xlabel("Permutation importance (R$^2$ drop)"); save("fig5_feature_importance")

In [13]:
# Fig6 WIP over time, Fig7 Gantt window
fig, ax = plt.subplots(figsize=(7.5, 2.8))
for nm in ["EDD", "AI-dispatch + AI-PdM"]:
    t, w_ = zip(*aux[nm]["wip"]); t = np.array(t); w_ = np.array(w_); k = (t > 2000)
    s = pd.Series(w_[k], index=t[k]).rolling(60, min_periods=1).mean(); ax.plot(s.index, s.values, color=C[nm], label=nm, lw=1.3)
ax.set(xlabel="Simulated time (min)", ylabel="Work-in-process (jobs)"); ax.legend(frameon=False); save("fig6_wip")
g = aux["AI-dispatch + AI-PdM"]["gantt"]; t0, t1 = 6000, 6700; fig, ax = plt.subplots(figsize=(8, 3))
rng = np.random.default_rng(1); jc = {}
for m, t, ex, dur, rep, j, act, f in g:
    if not (t0 <= t <= t1): continue
    col = jc.setdefault(j, plt.cm.tab20(len(jc) % 20))
    if ex: ax.barh(m, ex, left=t, color="#E69F00", height=.7, edgecolor="k", lw=.3)
    ax.barh(m, dur, left=t + ex, color=col, height=.7, edgecolor="w", lw=.3)
    if rep: ax.barh(m, rep, left=t + ex + dur, color="#D55E00", hatch="///", height=.7, edgecolor="k", lw=.3)
ax.set(xlim=(t0, t1), xlabel="Simulated time (min)", ylabel="Machine", yticks=range(6)); ax.invert_yaxis()
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color="#E69F00", label="AI-triggered intervention"), Patch(facecolor="#D55E00", hatch="///", label="Breakdown / repair")], frameon=False, fontsize=7, ncol=2, loc="upper center", bbox_to_anchor=(.5, 1.18)); save("fig7_gantt")
print("figures saved to", F)

figures saved to f:\Projects\smart_factory/figures


## 6. Extra factory instances

In [14]:
INSTANCES = {"A_baseline": {}, "B_large_8_machines": dict(n_mach=8), "C_harsh_failures": dict(wear_rate=0.3, repair=60.0, tool_change=25.0)}
N_INST = 2 if QUICK else 8
PC = {"EDD": "#0072B2", "ATC": "#332288", "EDD + periodic PM": "#E69F00", "ATC + AI-PdM": "#D55E00"}
P4 = {"EDD": dict(rule="EDD"), "ATC": dict(rule="ATC", kk=kk), "EDD + periodic PM": dict(rule="EDD", pdm="periodic", per_wear=pw),
      "ATC + AI-PdM": dict(rule="ATC", kk=kk, pdm="predictive", thr=thr)}
rows = [dict(instance=n, policy=p, seed=s, **simulate({**CFG, **ov}, s, 0.9, pol, fm)["res"])
        for n, ov in INSTANCES.items() for s in range(3000, 3000 + N_INST) for p, pol in P4.items()]
di = pd.DataFrame(rows); di.to_csv(f"{R}/instances_runs.csv", index=False)
gi = di.groupby(["instance", "policy"], sort=False)[["mean_tard", "pct_tardy", "mean_flow", "failures", "downtime"]].mean().round(1)
gi.to_csv(f"{R}/instances_summary.csv"); display(gi)
fig, ax = plt.subplots(figsize=(7, 3.4)); w = .2
for k, p in enumerate(P4):
    x = di[di.policy == p].groupby("instance", sort=False)["mean_tard"].agg(["mean", "std", "count"])
    ax.bar(np.arange(len(x)) + (k - 1.5) * w, x["mean"], w, yerr=1.96 * x["std"] / np.sqrt(x["count"]), color=PC[p], label=p, capsize=2)
ax.set_xticks(range(len(INSTANCES))); ax.set_xticklabels([i.replace("_", "\n", 1) for i in INSTANCES], fontsize=8)
ax.set_ylabel("Mean tardiness (min)"); ax.legend(frameon=False, fontsize=7); save("fig8_instances")

mean_tard  pct_tardy  mean_flow  \
instance           policy                                               
A_baseline         EDD                     56.7       37.3      377.6   
                   ATC                     32.7       30.3      349.5   
                   EDD + periodic PM       66.2       47.2      406.2   
                   ATC + AI-PdM             4.9        8.9      259.4   
B_large_8_machines EDD                     48.8       43.3      384.3   
                   ATC                     26.9       33.3      358.1   
                   EDD + periodic PM       64.2       46.1      405.7   
                   ATC + AI-PdM             4.2        8.6      273.8   
C_harsh_failures   EDD                    236.2       83.4      618.4   
                   ATC                    150.8       72.0      526.8   
                   EDD + periodic PM      344.1       91.6      735.4   
                   ATC + AI-PdM            25.5       28.1      340.0   

                                      failures  downtime  
instance           policy                                 
A_baseline         EDD                   178.5    7214.6  
                   ATC                   178.9    7144.7  
                   EDD + periodic PM     122.2    8517.9  
                   ATC + AI-PdM           60.0    3435.1  
B_large_8_machines EDD                   237.1    9513.0  
                   ATC                   234.6    9374.6  
                   EDD + periodic PM     159.4   11187.3  
                   ATC + AI-PdM           77.4    4525.9  
C_harsh_failures   EDD                   204.0   12048.1  
                   ATC                   206.5   12282.7  
                   EDD + periodic PM     119.2   15866.3  
                   ATC + AI-PdM           82.8    6792.6

## 7. Robustness

In [15]:
SCEN = {"nominal": {}, "tool life -20%": dict(life=.8), "tool life -40%": dict(life=.6), "torque +6 Nm": dict(torque_shift=6.0),
        "sensor noise 10%": dict(sensor_noise=.1), "sensor noise 30%": dict(sensor_noise=.3),
        "combined": dict(life=.7, torque_shift=4.0, sensor_noise=.2)}
N_ROB = 2 if QUICK else 8
PR = {"ATC": dict(rule="ATC", kk=kk), "ATC + periodic PM": dict(rule="ATC", kk=kk, pdm="periodic", per_wear=pw),
      "ATC + AI-PdM": dict(rule="ATC", kk=kk, pdm="predictive", thr=thr)}
rr = [dict(scenario=n, policy=p, seed=s, **simulate({**CFG, **ov}, s, 0.9, pol, fm)["res"])
      for n, ov in SCEN.items() for s in range(4000, 4000 + N_ROB) for p, pol in PR.items()]
dr = pd.DataFrame(rr); dr.to_csv(f"{R}/robustness_runs.csv", index=False)
pv = dr.pivot_table(index=["scenario", "seed"], columns="policy", values="mean_tard").reset_index()
out = []
for n in SCEN:
    q = pv[pv.scenario == n]
    for p in ["ATC + periodic PM", "ATC + AI-PdM"]:
        g = 100 * (1 - q[p] / q["ATC"]); out.append(dict(scenario=n, policy=p, reduction_vs_ATC_pct=g.mean(), ci95=1.96 * g.std(ddof=1) / np.sqrt(len(g))))
ro = pd.DataFrame(out).round(1); ro.to_csv(f"{R}/robustness_summary.csv", index=False); display(ro)
fig, ax = plt.subplots(figsize=(7.5, 3.4)); w = .38
for k, p in enumerate(["ATC + periodic PM", "ATC + AI-PdM"]):
    x = ro[ro.policy == p]; ax.bar(np.arange(len(x)) + (k - .5) * w, x.reduction_vs_ATC_pct, w, yerr=x.ci95, capsize=2, color=PC.get("EDD + periodic PM") if k == 0 else PC["ATC + AI-PdM"], label=p)
ax.axhline(0, color="k", lw=.6); ax.set_xticks(range(len(SCEN))); ax.set_xticklabels([s.replace(" ", "\n", 1) for s in SCEN], fontsize=7)
ax.set_ylabel("Tardiness reduction vs ATC (%)"); ax.legend(frameon=False, fontsize=7); save("fig9_robustness")

,scenario,policy,reduction_vs_ATC_pct,ci95
0,nominal,ATC + periodic PM,-18.6,35.7
1,nominal,ATC + AI-PdM,89.8,3.9
2,tool life -20%,ATC + periodic PM,-9.7,42.3
3,tool life -20%,ATC + AI-PdM,87.9,7.5
4,tool life -40%,ATC + periodic PM,29.5,15.4
5,tool life -40%,ATC + AI-PdM,84.5,5.0
6,torque +6 Nm,ATC + periodic PM,-34.5,23.4
7,torque +6 Nm,ATC + AI-PdM,95.2,2.9
8,sensor noise 10%,ATC + periodic PM,-18.6,35.7
9,sensor noise 10%,ATC + AI-PdM,57.5,25.9
